In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('face-matching-aicc-round-2')

path+='/face-matching/'
print("Path to competition files:", path)

Path to competition files: C:\Users\raian\.cache\kagglehub\competitions\face-matching-aicc-round-2/face-matching/


In [2]:
import pandas as pd
from PIL import Image
from tqdm import tqdm
import numpy as np
import cv2
import os

import torch
from transformers import CLIPProcessor, CLIPModel

device = 'cuda'

In [3]:
ref_df = pd.read_csv(path + "/ref_img.csv", dtype={"ref_img": str})
ref_ids = ref_df["ref_img"].tolist()
print(f"Reference IDs: {ref_ids}")

Reference IDs: ['048', '025', '095', '043', '105', '071', '046', '096', '020', '085', '061', '073', '084', '026', '008']


In [4]:
model_endpoint = 'openai/clip-vit-base-patch32'
processor = CLIPProcessor.from_pretrained(model_endpoint)
model = CLIPModel.from_pretrained(model_endpoint)

model.to(device)
model.eval()
model

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05, eleme

In [5]:
all_images = sorted([f[:-4] for f in os.listdir(f"{path}/images") if f.endswith(".jpg")])

In [7]:
print(type(model))

<class 'transformers.models.clip.modeling_clip.CLIPModel'>


In [10]:
features = {}
for img_name in tqdm(all_images):
    img_path = f"{path}/images/{img_name}.jpg"
    img = Image.open(img_path).convert("RGB")

    with torch.no_grad():
        inputs = processor(img, return_tensors='pt').to(device)
        feature = model.get_image_features(**inputs).pooler_output
        features[img_name] = feature.cpu().numpy()


100%|██████████| 109/109 [00:01<00:00, 58.38it/s]


In [ ]:
results = []

for ref_id in tqdm(ref_ids):
    ref_feature = features.get(ref_id)
    if ref_feature is None:
        continue

    similarities = {}
    for img_id, feature in features.items():
        sim = np.dot(ref_feature[0], feature[0]) / (
            np.linalg.norm(ref_feature[0]) * np.linalg.norm(feature[0]) + 1e-8
        )
        similarities[img_id] = float(sim)

    # sort by similarity, exclude reference image, take top 5
    sorted_ids = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    top_5 = [img_id for img_id, _ in sorted_ids if img_id != ref_id][:5]

    results.append({"ref_img": ref_id, "photos": "|".join(top_5)})



100%|██████████| 15/15 [00:00<00:00, 855.13it/s]


In [12]:
results

[{'ref_img': '048', 'photos': '101|102|066|021|078'},
 {'ref_img': '025', 'photos': '040|067|086|041|006'},
 {'ref_img': '095', 'photos': '021|097|014|047|072'},
 {'ref_img': '043', 'photos': '013|034|027|050|090'},
 {'ref_img': '105', 'photos': '058|024|108|029|082'},
 {'ref_img': '071', 'photos': '070|088|049|065|017'},
 {'ref_img': '046', 'photos': '060|082|010|093|019'},
 {'ref_img': '096', 'photos': '068|015|098|035|018'},
 {'ref_img': '020', 'photos': '031|053|074|008|087'},
 {'ref_img': '085', 'photos': '009|103|064|037|086'},
 {'ref_img': '061', 'photos': '104|019|107|084|002'},
 {'ref_img': '073', 'photos': '090|039|003|082|055'},
 {'ref_img': '084', 'photos': '012|089|016|081|055'},
 {'ref_img': '026', 'photos': '051|099|080|067|041'},
 {'ref_img': '008', 'photos': '000|087|020|074|054'}]